In [1]:
!pip -q install transformers accelerate pillow requests > /dev/null

In [2]:
import torch
from PIL import Image
import requests
from io import BytesIO
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

NSFW_MODEL = "Falconsai/nsfw_image_detection"
VIOLENCE_MODEL = "jaranohaal/vit-base-violence-detection"

nsfw_clf = pipeline("image-classification", model=NSFW_MODEL, device=device)
violence_clf = pipeline("image-classification", model=VIOLENCE_MODEL, device=device)

print("Loaded models:", NSFW_MODEL, "and", VIOLENCE_MODEL)


Using device: CPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Some weights of the model checkpoint at jaranohaal/vit-base-violence-detection were not used when initializing ViTForImageClassification: ['blocks.0.attn.proj.bias', 'blocks.0.attn.proj.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc2.bias', 'blocks.0.mlp.fc2.weight', 'blocks.0.norm1.bias', 'blocks.0.norm1.weight', 'blocks.0.norm2.bias', 'blocks.0.norm2.weight', 'blocks.1.attn.proj.bias', 'blocks.1.attn.proj.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.mlp.fc1.bias', 'blocks.1.mlp.fc1.weight', 'blocks.1.mlp.fc2.bias', 'blocks.1.mlp.fc2.weight', 'blocks.1.norm1.bias', 'blocks.1.norm1.weight', 'blocks.1.norm2.bias', 'blocks.1.norm2.weight', 'blocks.10.attn.proj.bias', 'blocks.10.attn.proj.weight', 'blocks.10.attn.qkv.bias', 'blocks.10.attn.qkv.weight', 'blocks.10.mlp.fc1.bias', 'blocks.10.mlp.fc1.weight', 'blocks.10.mlp.fc2.bias', 'blocks.10.mlp.fc2.weight', 'blocks.10.norm1.bi

preprocessor_config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
Device set to use cpu


Loaded models: Falconsai/nsfw_image_detection and jaranohaal/vit-base-violence-detection


In [3]:
def load_image_from_url(url: str) -> Image.Image:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")

def load_image_from_path(path: str) -> Image.Image:
    return Image.open(path).convert("RGB")


In [4]:
def top_label_and_score(preds):
    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    return preds[0]["label"], float(preds[0]["score"])

def decide_accept_reject(
    img: Image.Image,
    nsfw_reject_threshold: float = 0.50,
    violence_reject_threshold: float = 0.50,
    verbose: bool = True
):
    nsfw_preds = nsfw_clf(img)
    violence_preds = violence_clf(img)

    nsfw_label, nsfw_score = top_label_and_score(nsfw_preds)
    violence_label, violence_score = top_label_and_score(violence_preds)

    nsfw_label_l = nsfw_label.strip().lower()
    violence_label_l = violence_label.strip().lower()

    nsfw_reject = (nsfw_label_l == "nsfw") and (nsfw_score >= nsfw_reject_threshold)

    violence_reject = (("violent" in violence_label_l) or ("violence" in violence_label_l)) and \
                      (("non" not in violence_label_l) and (violence_score >= violence_reject_threshold))

    decision = "REJECTED" if (nsfw_reject or violence_reject) else "ACCEPTED"

    result = {
        "decision": decision,
        "nsfw": {"label": nsfw_label, "score": nsfw_score, "all": nsfw_preds},
        "violence": {"label": violence_label, "score": violence_score, "all": violence_preds},
        "thresholds": {
            "nsfw_reject_threshold": nsfw_reject_threshold,
            "violence_reject_threshold": violence_reject_threshold
        }
    }

    if verbose:
        print("Decision:", decision)
        print(f"NSFW -> label={nsfw_label}, score={nsfw_score:.4f} (reject if label=='nsfw' and score>={nsfw_reject_threshold})")
        print(f"Violence -> label={violence_label}, score={violence_score:.4f} (reject if violent and score>={violence_reject_threshold})")

    return result


In [15]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded.keys()))
img = load_image_from_path(image_path)

_ = decide_accept_reject(img)


Saving WhatsApp Image 2026-01-25 at 16.27.00.jpeg to WhatsApp Image 2026-01-25 at 16.27.00.jpeg
Decision: REJECTED
NSFW -> label=nsfw, score=0.9998 (reject if label=='nsfw' and score>=0.5)
Violence -> label=LABEL_1, score=0.6404 (reject if violent and score>=0.5)
